In [95]:
import pandas as pd
import matplotlib.pyplot as plt

import random
import re

In [96]:
import pandas as pd

def load_dictionaries(file_path="Data_Teencode.xlsx"):
    # Đọc sheet Normalization
    df_n = pd.read_excel(file_path, sheet_name="Normalization")
    norm_map = dict(zip(df_n['Từ_viết_tắt'], df_n['Dạng_chuẩn']))
    
    # Đọc sheet Noise
    df_noise = pd.read_excel(file_path, sheet_name="Noise")
    noise_dict = {}
    for _, row in df_noise.iterrows():
        # Chuyển chuỗi "k, ko, khum" thành list ["k", "ko", "khum"]
        noise_dict[str(row['Từ_chuẩn'])] = [v.strip() for v in str(row['Biến_thể']).split(",")]
        
    return norm_map, noise_dict

# Load lại để sử dụng
normalization_map, noise_dict = load_dictionaries()

In [97]:
print(normalization_map, noise_dict)

{'acp': 'đồng ý', 'rep': 'trả lời', 'off': 'ngoại tuyến', 'onl': 'trực tuyến', 'avt': 'ảnh đại diện', 'unf': 'hủy kết bạn', 'FB': 'Facebook', 'ig': 'Instagram', 'tt': 'Tiktok', 'xl': 'Zalo', 'dép lào': 'Zalo', 'mh': 'mấy giờ', 'mess': 'Messenger', 'fl': 'theo dõi', 'sđt': 'số điện thoại', 'yt': 'YouTube', 'ytb': 'YouTube', 'cmt': 'bình luận', 'acc': 'tài khoản', 'cmsn': 'chúc mừng sinh nhật', 'cmnm': 'chúc mừng năm mới', 'SK': 'sức khỏe', 'sk': 'sự kiện', 'BV': 'bệnh viện', 'ib': 'nhắn tin', 'in4': 'thông tin', 'ykr': 'ý kiến riêng', 'mem': 'thành viên', 'mxh': 'mạng xã hội', 'otp': 'ghép cặp', 'scam': 'lừa gạt', 'slay': 'xuất sắc', 'stt': 'trạng thái', 'tag': 'gắn thẻ', 'tcn': 'trang cá nhân', 'toxic': 'tiêu cực', 'cr': 'người mình thích', 'cre': 'nguồn'} {'bạn': ['bn', 'b', 'bro'], 'mình': ['mìn', 'mik', 'mk'], 'bình luận': ['bl'], 'bao cao su': ['bcs'], 'quan trọng': ['qtrong', 'qtr'], 'thời gian': ['tgian', 'tg'], 'ví dụ': ['vd', 'vdu'], 'người ta': ['ngta'], 'trả lời': ['trl'], 'b

In [98]:

global_missing_keys = set()

In [99]:
# ==========================================
# HÀM 2: RẢI NHIỄU VÀO TẬP VILEXNORM SẠCH
# ==========================================
def inject_noise(clean_text, min_noise=1, max_noise=3):
    global global_missing_keys
    if not isinstance(clean_text, str): return ""
    
    words = clean_text.split()
    new_words = words.copy()
    
    # Tìm tất cả các từ/cụm từ trong câu có khả năng bị biến thành nhiễu
    candidates = []
    i = 0
    while i < len(words):
        matched = False
        for root_word in sorted_noise_keys:
            root_len = len(root_word.split())
            if i + root_len <= len(words):
                phrase = " ".join(words[i:i+root_len]).lower()
                if phrase == root_word:
                    candidates.append((i, root_len, root_word))
                    i += root_len
                    matched = True
                    break
        if not matched:
            i += 1

    # Quyết định số lượng nhiễu (từ 1 đến 3)
    num_noise = min(random.randint(min_noise, max_noise), len(candidates))
    
    if num_noise > 0:
        # Chọn ngẫu nhiên các vị trí để chèn nhiễu
        selected_candidates = random.sample(candidates, num_noise)
        
        # Thay thế từ cuối câu lên đầu câu để không làm hỏng index
        selected_candidates.sort(key=lambda x: x[0], reverse=True)
        
        for idx, length, root_word in selected_candidates:
            # Kiểm tra key trước khi lấy variant
            if root_word not in noise_dict:
                global_missing_keys.add(root_word)
                variant = root_word
            else:
                variant = random.choice(noise_dict[root_word])
            # Thực hiện replace
            new_words[idx:idx+length] = [variant]
            
    return " ".join(new_words)

In [100]:
def process_vilexnorm(df, noise_prob=0.7):
    processed_df = df.copy()
    
    # Đảm bảo có cột cần thiết
    required = ['original', 'normalized']
    for col in required:
        if col not in processed_df.columns:
            raise ValueError(f"Thiếu cột {col}")

    def apply_strategy(row):
        target = str(row['normalized'])
        # 70% trường hợp: Bơm nhiễu mạnh tay
        if random.random() < noise_prob:
            # Sạch hóa cụm từ tắt cứng -> Bơm nhiễu linh hoạt
            return inject_noise(normalize_pre_train(target))
        else:
            # 30% trường hợp: Giữ nhiễu tự nhiên của ViLexNorm
            return str(row['original'])

    processed_df['source'] = processed_df.apply(apply_strategy, axis=1)
    
    # Xuất kết quả cuối cùng
    final_df = processed_df[['source', 'normalized']].copy()
    final_df.columns = ['source', 'target']
    return final_df

# CHẠY THỰC TẾ
# df2 = process_vilexnorm(df)
# print(df2.head())

In [ ]:
import pandas as pd

# Lấy dữ liệu từ file CSV online
url = 'https://raw.githubusercontent.com/<username>/<repo>/main/ViLexNorm.csv'  # Thay <username> và <repo> cho đúng

df = pd.read_csv(url, encoding='utf-8')
df.head()

,original,normalized,input,output
0,thích anh cá mập k,thích anh cá mập không,"['thích', 'anh', 'cá', 'mập', 'k']","['thích', 'anh', 'cá', 'mập', 'không']"
1,cứ ngây thơ thế thoai :)),cứ ngây thơ thế thôi :)),"['cứ', 'ngây', 'thơ', 'thế', 'thoai', ':))']","['cứ', 'ngây', 'thơ', 'thế', 'thôi', ':))']"
2,bà nghê xinh vậy mà t thấy k bằng bà chipu luô...,bà nghê xinh vậy mà tôi thấy không bằng bà chi...,"['bà', 'nghê', 'xinh', 'vậy', 'mà', 't', 'thấy...","['bà', 'nghê', 'xinh', 'vậy', 'mà', 'tôi', 'th..."
3,ê k khóc được làm thế nào má =)) ?,ê không khóc được làm thế nào má =)) ?,"['ê', 'k', 'khóc', 'được', 'làm', 'thế', 'nào'...","['ê', 'không', 'khóc', 'được', 'làm', 'thế', '..."
4,có biến gì hong dẫy :)),có biến gì không vậy :)),"['có', 'biến', 'gì', 'hong', 'dẫy', ':))']","['có', 'biến', 'gì', 'không', 'vậy', ':))']"


In [102]:
df2 = process_vilexnorm(df)
df2.head()

,source,target
0,thích a cá mập khum,thích anh cá mập không
1,cứ ngây thơ thế thui :)),cứ ngây thơ thế thôi :))
2,bà nghê xinh dậy mà tôi thấy ko bằng bà chipu ...,bà nghê xinh vậy mà tôi thấy không bằng bà chi...
3,ê k khóc được làm thế nào má =)) ?,ê không khóc được làm thế nào má =)) ?
4,có biến j khom dậy :)),có biến gì không vậy :))


In [103]:
# In tổng số key bị thiếu sau khi xử lý xong
print(f"Tổng số key bị thiếu: {len(global_missing_keys)}")
print(f"Danh sách key bị thiếu: {global_missing_keys}")

Tổng số key bị thiếu: 3
Danh sách key bị thiếu: {'cười', 'người mình thích', 'nguồn'}
